# Dual-hose inference (L, R) — folder: `dual_hose_tracker`

Writes `hoseL_keypoints.csv` and `hoseR_keypoints.csv`.
Uses `unet_efficientnet_hose_lr_checkpoint.weights.h5`.

For L+R+T+B inference see `../quad_hose_tracker/`.


In [1]:
import sys, importlib
sys.path.append('../../..')
sys.path.append('/home/wanglab/Programs/tracking/DeepLearningUtils')
sys.path.append('/home/wanglab/Programs/tracking/DeepLearningUtils/src')
sys.path.append('/home/wanglab/src/keras-unet-collection')

from keras_unet_collection import models
import keras
import hose_data
importlib.reload(hose_data)
from hose_data import process_video_batch_dual, KEYPOINT_NAMES
print('channels:', KEYPOINT_NAMES)


I0000 00:00:1787776228.799955  146312 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


channels: ('hoseL', 'hoseR')


In [2]:
video_files = [
    "/mnt/c/Users/wanglab/Desktop/TG/NTNG1-PrV-Piezo/P005/102225_2/output.mp4",
]

batch_size = 32
input_shape = (256, 256, 3)
video_hw = (480, 640)

checkpoint_path = 'unet_efficientnet_hose_lr_checkpoint.weights.h5'
threshold = 0.65
require_both = True


In [3]:
filter_num = [64, 64, 64, 64, 64, 64]
base = models.att_unet_2d(
    input_shape, filter_num=filter_num, n_labels=2,
    stack_num_down=2, stack_num_up=2, activation='ReLU', atten_activation='ReLU',
    attention='add', output_activation='Sigmoid', batch_norm=True, dropout=True,
    dropout_rate=0.1, l2_regularization=False, l2_weight=1e-4, pool=False, unpool=False,
    backbone='EfficientNetB1', weights='imagenet',
    freeze_backbone=True, freeze_batch_norm=True, name='attunet',
)
inputs = keras.Input(shape=input_shape)
x = keras.applications.efficientnet.preprocess_input(inputs)
model = keras.Model(inputs, base(x), name='dual_hose_tracker')
model.load_weights(checkpoint_path)

inputs_vid = keras.Input(shape=(video_hw[0], video_hw[1], 3))
x = keras.ops.cast(inputs_vid, dtype='float32')
x = keras.layers.Resizing(height=input_shape[0], width=input_shape[1])(x)
model = keras.Model(inputs_vid, model(x))


I0000 00:00:1787776236.511306  146312 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13360 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Ti, pci bus id: 0000:04:00.0, compute capability: 12.0a


In [4]:
if not video_files:
    raise ValueError('Add video paths (with .mp4).')
process_video_batch_dual(
    video_paths=video_files, model=model, batch_size=batch_size,
    threshold=threshold, require_both=require_both,
)


Video size: 640.0 x 480.0
Total number of frames in video 600063


Video 1/1:   0%|          | 0/600063 [00:00<?, ?it/s]I0000 00:00:1787776245.091691  156030 service.cc:153] XLA service 0x7ae7e009c730 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787776245.091737  156030 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5060 Ti, Compute Capability 12.0a (Driver: 13.3.0; Runtime: 12.8.0; Toolkit: 12.8.0; DNN: 9.10.2)
I0000 00:00:1787776245.337368  156030 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787776246.770435  156030 cuda_dnn.cc:461] Loaded cuDNN version 91002
E0000 00:00:1787776250.123480  156030 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1787776257.286710  156030 cuda_timer.cc:87] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup exe


Processing complete. Log file: /mnt/c/Users/wanglab/Desktop/TG/NTNG1-PrV-Piezo/P005/102225_2/hose_lr_processing_20260826_163042.log
